# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

**Execution notes.** This is the supplied template with its original task order and
instructions retained. Run top to bottom. The configuration supports CUDA, Apple MPS
and CPU; full fine-tuning is best run on a GPU. `QUICK_RUN=True` selects a small subset.
Full-run defaults remain 5,000 training and 400 test examples. Outputs are empty until execution.


In [1]:
# Install the packages in this notebook's Python kernel; version bounds keep the APIs compatible.
# After the first installation, restart the kernel if Jupyter asks you to.
%pip install -q "torch>=2.6,<3" "transformers==4.57.1" "datasets>=3,<5" "accelerate>=1,<2" "nltk>=3.9,<4" "pandas>=2,<4" "matplotlib>=3.8,<4" "tqdm>=4.66" "evaluate>=0.4,<1" "rouge-score==0.1.2"



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [2]:
# Import the libraries and configure the base model, dataset sizes, and fine-tuning runs.
import gc
import json
import math
import time
import zipfile
from pathlib import Path
import pandas as pd
import torch
from torch import nn
from datasets import Dataset, DatasetDict, load_dataset
from rouge_score.rouge_scorer import RougeScorer
from IPython.display import display, Markdown, FileLink
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, set_seed

# True uses a small dataset for debugging; False uses the requested 5,000/400 examples.
QUICK_RUN = False
FORCE_CPU = False
SEED = 101
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'
MAX_TRAIN_SAMPLES = 64 if QUICK_RUN else 5000
MAX_TEST_SAMPLES = 16 if QUICK_RUN else 400
# Maximum prompt + response length in tokens, including the response EOS token.
MAX_LENGTH = 512
EPOCHS = 1
# Accumulate 8 batches of size 1 per optimizer update to reduce peak memory on a single device.
BATCH_SIZE, EVAL_BATCH_SIZE, GRAD_ACCUM = 1, 1, 8
# Full SFT and LoRA use different learning rates here; mention this when interpreting their comparison.
FULL_LEARNING_RATE, LORA_LEARNING_RATE = 5e-5, 2e-4
# Rank controls adapter size; alpha/r scales the adapter contribution.
LORA_RANK, LORA_ALPHA = 8, 16
QUALITATIVE_TEST_SAMPLES = 5
MAX_NEW_TOKENS = 128
CUSTOM_QUESTIONS = [
    'Explain gravity in one short sentence.',
    'What is the capital of Sweden?',
    'Write a Python function that reverses a list.',
]
# Save checkpoints/adapters here for the later reload step; metric exports are optional copies.
OUTPUT_DIR = Path('outputs') / ('assignment3_quick' if QUICK_RUN else 'assignment3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Choose CUDA, Apple MPS or CPU according to availability and FORCE_CPU.
DEVICE = torch.device('cpu' if FORCE_CPU else 'cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available() else 'cpu')
# Enable reduced-precision training only on CUDA; use BF16 when supported, otherwise FP16.
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
USE_FP16 = DEVICE.type == 'cuda' and not USE_BF16
# Seed model initialization and data shuffling for repeatable runs on the same setup.
set_seed(SEED)
pd.set_option('display.max_colwidth', 140)
print('Device:', DEVICE, '| Mode:', 'small subset' if QUICK_RUN else 'full experiment')


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps | Mode: full experiment


In [3]:
# Optional bookkeeping: save the settings so a later result can be traced to its configuration.
# This JSON file is a convenience, not a required assignment deliverable.
run_config = {
    'quick_run': QUICK_RUN, 'base_model': MODEL_NAME, 'seed': SEED,
    'train_samples': MAX_TRAIN_SAMPLES, 'test_samples': MAX_TEST_SAMPLES,
    'max_length': MAX_LENGTH, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
    'gradient_accumulation': GRAD_ACCUM, 'full_lr': FULL_LEARNING_RATE,
    'lora_lr': LORA_LEARNING_RATE, 'rank': LORA_RANK, 'alpha': LORA_ALPHA,
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2))


287

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [4]:
# Read the dataset as a stream; filtering and take() below select the examples we actually need.
# Streaming avoids downloading the entire collection before selecting the subset.
smoltalk = load_dataset('HuggingFaceTB/smoltalk', 'all', streaming=True)


In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [5]:
# Keep short single-turn conversations, with an optional system message, then select train/test subsets.
def eligible_example(row):
    messages = row['messages']
    roles = [message['role'] for message in messages]
    # Allow user-assistant pairs with at most one preceding system message; lengths here are characters.
    return (roles in (['user', 'assistant'], ['system', 'user', 'assistant'])
            and all(len(message['content']) <= 256 for message in messages))

filtered = smoltalk.filter(eligible_example)
selected = {}
# Select each existing split independently; test examples are not moved into the training set.
for split, limit in [('train', MAX_TRAIN_SAMPLES), ('test', MAX_TEST_SAMPLES)]:
    # Materialize only the selected examples into a regular Dataset for Trainer.
    rows = [{'messages': row['messages']} for row in filtered[split].take(limit)]
    if len(rows) != limit:
        raise ValueError(f'Only {len(rows)} eligible {split} rows; requested {limit}.')
    selected[split] = Dataset.from_list(rows)
smoltalk_simplified = DatasetDict(selected)


In [6]:
# Display the selected dataset splits and their sizes.
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [7]:
# Inspect one conversation to see its message roles and text.
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}]}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [8]:
# Separate the conversation prefix from the assistant answer that the model must learn to generate.
def format_input_output(example):
    messages = example['messages']
    if not messages or messages[-1]['role'] != 'assistant':
        raise ValueError('The last message must be the target assistant response.')
    headings = {'system': 'System', 'user': 'User', 'assistant': 'Assistant'}
    # Include earlier system/user messages as context; exclude the final assistant answer from the prompt.
    prompt = ''.join(f"{headings[m['role']]}: {m['content'].strip()}\n\n" for m in messages[:-1])
    # End the prompt at the Assistant heading so training and generation share the same boundary.
    return {'prompt': prompt + 'Assistant:\n', 'response': messages[-1]['content'].strip()}


Apply the function you implemented to the dataset as a whole.

In [9]:
# Apply the formatting function to every example in both dataset splits.
ds_sft = smoltalk_simplified.map(format_input_output)

Map: 100%|██████████| 400/400 [00:00<00:00, 51592.04 examples/s]


Then verify that the dataset now contains the new fields you created.

In [10]:
# Inspect one formatted prompt/response pair before converting it to token IDs.
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'prompt': "System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\n\nUser: Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.\n\nAssistant:\n",
 'response': 'The chef made more food after the restaurant ran out.'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [11]:
# Use the pretrained model's own tokenizer so token IDs match its learned embeddings.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.eos_token_id is None:
    raise ValueError('Choose a model with an EOS token.')
# Reuse EOS as a padding ID if necessary; masks distinguish actual EOS targets from added padding.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('EOS:', tokenizer.eos_token_id, '| PAD:', tokenizer.pad_token_id)


EOS: 0 | PAD: 0


Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [12]:
# Build the input sequence and train only on the assistant response, conditioned on the prompt.
def tokenize_helper(example):
    max_length = MAX_LENGTH
    # Encode the prefix separately so its token boundary matches prompted generation.
    prefix = tokenizer.encode(example['prompt'], add_special_tokens=False)
    response = tokenizer.encode(example['response'], add_special_tokens=False)
    if len(prefix) >= max_length - 1:
        raise ValueError('Prompt leaves no response space; increase --max-length.')
    # Truncate the response if necessary but reserve its final EOS, teaching the model when to stop.
    response = response[:max_length - len(prefix) - 1] + [tokenizer.eos_token_id]
    # Prompt tokens remain visible as context. Labels of -100 exclude their prediction from the loss.
    # All positions are real tokens here; the collator later appends padding and zeros in the attention mask.
    return {'input_ids': prefix + response,
            'attention_mask': [1] * (len(prefix) + len(response)),
            'labels': [-100] * len(prefix) + response}


As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.

In [13]:
# Tokenize both splits and verify that the supervised labels contain the response only.
# Remove the original text columns so batches contain only input_ids, labels and attention_mask.
tokenized_ds_sft = ds_sft.map(tokenize_helper, remove_columns=ds_sft['train'].column_names)
example = tokenized_ds_sft['train'][0]
assert len(example['input_ids']) == len(example['labels']) == len(example['attention_mask'])
assert any(label == -100 for label in example['labels'])
assert any(label != -100 for label in example['labels'])
print('Prompt and response:')
print(tokenizer.decode(example['input_ids']))
print()
print('Supervised response only:')
# Decode only non-masked targets to confirm that the prompt is not being supervised.
print(tokenizer.decode([label for label in example['labels'] if label != -100]))


Map: 100%|██████████| 400/400 [00:00<00:00, 6144.96 examples/s]

Prompt and response:
System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.

User: Rearrange this sentence to make it easy to understand:
The restaurant ran out of food, so the chef made some more.

Assistant:
The chef made more food after the restaurant ran out.<|endoftext|>

Supervised response only:
The chef made more food after the restaurant ran out.<|endoftext|>



## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [14]:
# Combine unequal-length examples into rectangular tensors with shape [batch, longest sequence].
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    # Token IDs and labels must be integer tensors, as expected by embeddings and cross-entropy.
    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    # Pad only to this batch's maximum length to avoid unnecessary work on shorter batches.
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        # Stack the padded rows into one [batch, sequence] tensor.
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    # Mask added padding with -100 without masking the real response EOS token.
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

In [15]:
# Check that inputs, masks and labels align and that every padding position is excluded from loss.
batch = data_collator([tokenized_ds_sft['train'][0], tokenized_ds_sft['train'][1]])
assert batch['input_ids'].shape == batch['labels'].shape == batch['attention_mask'].shape
assert torch.all(batch['labels'][batch['attention_mask'] == 0] == -100)
print({name: tuple(value.shape) for name, value in batch.items()})


{'input_ids': (2, 79), 'attention_mask': (2, 79), 'labels': (2, 79)}


The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

**Metric correction.** The helper below aligns each prediction at position $t$ with
its target at $t+1$. It scores only the supervised response span and reduces logits
to token IDs before accumulation. It uses `rouge_score` directly and does not assume
a model-specific end token. This is **teacher-forced ROUGE-L**: the correct prefix
remains available. Freely generated answers are examined separately in Task 4.4.


In [16]:
# Compute response-only ROUGE-L from the model's next-token predictions.
# This teacher-forced score uses gold preceding tokens; it is different from free-running generation.
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    # Retain only the highest-scoring token ID at each position instead of storing full vocabulary logits.
    return logits.argmax(dim=-1)


class RougeMetricComputer:
    """Response-only, next-token-aligned teacher-forced ROUGE-L."""
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.scorer = RougeScorer(['rougeL'], use_stemmer=True)
        self.total, self.count = 0.0, 0

    def __call__(self, eval_pred, compute_result=False):
        predicted = torch.as_tensor(eval_pred.predictions)
        labels = torch.as_tensor(eval_pred.label_ids)
        # logits at position t predict the label at t+1, not the label at t.
        for pred, gold in zip(predicted[:, :-1], labels[:, 1:]):
            # Score only assistant-response targets, excluding prompt and padding positions.
            mask = gold != -100
            if not mask.any():
                continue
            predicted_text = self.tokenizer.decode(pred[mask].tolist(), skip_special_tokens=True)
            gold_text = self.tokenizer.decode(gold[mask].tolist(), skip_special_tokens=True)
            # ROUGE-L F1 measures overlap through the longest common subsequence of predicted and reference text.
            self.total += self.scorer.score(gold_text, predicted_text)['rougeL'].fmeasure
            self.count += 1
        # At the final evaluation batch, return the mean per-example score and reset the accumulator.
        if compute_result:
            result = {'rougeL_teacher_forced': self.total / self.count if self.count else 0.0}
            self.total, self.count = 0.0, 0
            return result
        return {}


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [17]:
# Build identical data/metric plumbing for all variants and define training, timing and logging helpers.
def make_trainer(model, training_args):
    # Connect the train/test datasets, custom padding function and a fresh metric accumulator.
    return Trainer(
        model=model, args=training_args,
        train_dataset=tokenized_ds_sft['train'], eval_dataset=tokenized_ds_sft['test'],
        data_collator=data_collator, compute_metrics=RougeMetricComputer(tokenizer),
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )

# Use the same training setup for both tuned variants, apart from their chosen learning rate.
def make_training_args(variant, learning_rate):
    return TrainingArguments(
        output_dir=str(OUTPUT_DIR / variant), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=learning_rate,
        weight_decay=0.01, warmup_ratio=0.03, optim='adamw_torch',
        # Evaluate explicitly after training; save the final model/adapter explicitly below.
        eval_strategy='no', save_strategy='no', logging_steps=50,
        bf16=USE_BF16, fp16=USE_FP16, use_cpu=FORCE_CPU, report_to='none',
        # Accumulate ROUGE batch by batch to avoid keeping all evaluation predictions in memory.
        batch_eval_metrics=True, eval_accumulation_steps=1, seed=SEED, data_seed=SEED,
        label_names=['labels'],
    )

# After model references are deleted, reclaim unused Python objects and cached CUDA memory.
def release_cached_memory():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

# Wait for queued GPU work before reading a timing result.
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elif DEVICE.type == 'mps':
        torch.mps.synchronize()

# Display measurements and optionally keep a JSON copy for later reference.
def save_metrics(variant, metrics):
    directory = OUTPUT_DIR / variant
    directory.mkdir(exist_ok=True)
    (directory / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    display(pd.DataFrame([metrics]))


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [18]:
# Evaluate the untouched pretrained model to establish the baseline before any fine-tuning.
set_seed(SEED)
pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
pretrained_trainer = make_trainer(pretrained_model, make_training_args('base', FULL_LEARNING_RATE))
start = time.perf_counter()
# evaluate() runs inference and computes metrics; it does not optimize the model.
pretrained_eval_metrics = pretrained_trainer.evaluate()
synchronize()
pretrained_eval_time = time.perf_counter() - start
base_metrics = {
    'variant': 'base', 'parameters_updated': 0,
    'total_parameters': sum(p.numel() for p in pretrained_model.parameters()),
    'training_seconds': 0.0, 'evaluation_seconds': pretrained_eval_time,
    **pretrained_eval_metrics,
}
save_metrics('base', base_metrics)
# Release this model so the next variant does not compete for memory.
del pretrained_trainer, pretrained_model
release_cached_memory()


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


,variant,parameters_updated,total_parameters,training_seconds,evaluation_seconds,eval_loss,eval_model_preparation_time,eval_rougeL_teacher_forced,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,base,0,134515008,0.0,14.800991,1.89582,0.0016,0.603467,14.7988,27.029,27.029


A base model can score well under teacher forcing because language-model pretraining
already teaches next-token prediction and evaluation supplies the gold prefix.
This score alone does not show that the model follows instructions during free generation.



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [19]:
# Start from the pretrained checkpoint again and update all parameters with supervised fine-tuning.
set_seed(SEED)
# Use a fresh pretrained model, with all original weights trainable.
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
baseline_training_args = make_training_args('full', FULL_LEARNING_RATE)
baseline_trainer = make_trainer(base_model, baseline_training_args)
synchronize()
start = time.perf_counter()
# Trainer runs forward passes, masked causal loss, backpropagation and optimizer updates.
baseline_trainer.train()
synchronize()
full_training_time = time.perf_counter() - start
start = time.perf_counter()
full_eval_metrics = baseline_trainer.evaluate()
synchronize()
full_evaluation_time = time.perf_counter() - start
full_metrics = {
    'variant': 'full', 'training_seconds': full_training_time,
    'evaluation_seconds': full_evaluation_time, **full_eval_metrics,
}
# Keep the trained weights and matching tokenizer for the later generation comparison.
base_model.save_pretrained(OUTPUT_DIR / 'full')
tokenizer.save_pretrained(OUTPUT_DIR / 'full')
display(pd.DataFrame([full_metrics]))


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,1.414000
100,1.220200
150,1.249000
200,1.131500
250,1.146800
300,1.170300
350,1.122800
400,1.127900
450,1.070900
500,1.111500


,variant,training_seconds,evaluation_seconds,eval_loss,eval_rougeL_teacher_forced,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,full,1050.651106,17.651696,1.173011,0.67716,17.6486,22.665,22.665,1.0


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [20]:
# Count scalar parameters that receive gradients; frozen weights do not count as trainable.
def num_trainable_parameters(model):
    # numel counts scalar entries; requires_grad identifies parameters eligible for optimization.
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


Apply this function to the SFT-trained model and check that the result makes sense.

In [21]:
# Record the full model's parameter count, then release it before constructing the LoRA model.
full_metrics['parameters_updated'] = num_trainable_parameters(base_model)
full_metrics['total_parameters'] = sum(p.numel() for p in base_model.parameters())
save_metrics('full', full_metrics)
del baseline_trainer, base_model
release_cached_memory()


,variant,training_seconds,evaluation_seconds,eval_loss,eval_rougeL_teacher_forced,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch,parameters_updated,total_parameters
0,full,1050.651106,17.651696,1.173011,0.67716,17.6486,22.665,22.665,1.0,134515008,134515008


## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [22]:
# Find the attention projection layers that will receive low-rank trainable updates.
def extract_lora_targets(model):
    # Task 4.3 asks for all four attention projections, not only query and value.
    names = {'q_proj', 'k_proj', 'v_proj', 'o_proj'}
    # Keep linear q/k/v/output projections inside self-attention; leave MLP layers unwrapped.
    targets = {name: layer for name, layer in model.named_modules()
               if isinstance(layer, nn.Linear) and '.self_attn.' in name
               and name.rsplit('.', 1)[-1] in names}
    if not targets:
        raise ValueError('No supported q/k/v/o attention projections found in this model.')
    return targets


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [23]:
# Replace nested model layers using their dotted names, preserving the surrounding architecture.
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        # Walk to the parent module, then replace its final named child with the supplied layer.
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [24]:
# Wrap a frozen linear layer with a low-rank update: y = W(x) + (alpha/r) * B(A(x)).
# Only A and B are trained; their parameter count is r * (input_width + output_width).
class LoRALayer(nn.Module):
    def __init__(self, W, r=8, alpha=16):
        super().__init__()
        if not isinstance(W, nn.Linear) or r < 1:
            raise ValueError('LoRA needs an nn.Linear base and positive rank.')
        self.base_layer = W
        # Freeze the original matrix and any original bias.
        self.base_layer.requires_grad_(False)
        self.scaling = alpha / r
        # A reduces the input to rank r; B maps that compact representation to the output width.
        self.A = nn.Linear(W.in_features, r, bias=False, device=W.weight.device, dtype=W.weight.dtype)
        self.B = nn.Linear(r, W.out_features, bias=False, device=W.weight.device, dtype=W.weight.dtype)
        # Initialize A randomly but B at zero, making the initial adapter contribution exactly zero.
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    # Expose the original weight for compatibility with code that inspects linear layers.
    @property
    def weight(self):
        return self.base_layer.weight

    @property
    def bias(self):
        return self.base_layer.bias

    def forward(self, x):
        # Add the learned low-rank correction to the unchanged base-layer output.
        return self.base_layer(x) + self.scaling * self.B(self.A(x))


In [25]:
# Check that a newly initialized LoRA layer matches its base layer and has the expected trainable size.
test_linear = nn.Linear(8, 6)
test_input = torch.randn(2, 8)
expected_output = test_linear(test_input).detach()
test_lora = LoRALayer(test_linear, r=2, alpha=4)
torch.testing.assert_close(test_lora(test_input), expected_output)
assert not test_lora.base_layer.weight.requires_grad
# For input 8, output 6 and rank 2, the two matrices have 2*8 + 6*2 trainable entries.
assert num_trainable_parameters(test_lora) == 2 * (8 + 6)
print('LoRA starts with the same output as its frozen base layer.')


LoRA starts with the same output as its frozen base layer.


Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [26]:
# Start LoRA from the same original pretrained model and train only the added adapter weights.
set_seed(SEED)
lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# Freeze the entire pretrained model before inserting newly trainable LoRA A/B matrices.
lora_model.requires_grad_(False)
targets = extract_lora_targets(lora_model)
# The expected trainable count is the sum of r*(input+output) over the selected projections.
expected_count = sum(LORA_RANK * (layer.in_features + layer.out_features) for layer in targets.values())
# Wrap every selected q/k/v/output projection; all other weights remain frozen.
replace_layers(lora_model, {name: LoRALayer(layer, LORA_RANK, LORA_ALPHA)
                            for name, layer in targets.items()})
lora_parameter_count = num_trainable_parameters(lora_model)
# Check that only the intended adapter parameters are trainable.
assert lora_parameter_count == expected_count
print('Wrapped attention projections:', len(targets))
print('Trainable adapter parameters:', lora_parameter_count)

# Train on the same examples and masked objective as full SFT.
lora_trainer = make_trainer(lora_model, make_training_args('lora', LORA_LEARNING_RATE))
synchronize()
start = time.perf_counter()
lora_trainer.train()
synchronize()
lora_training_time = time.perf_counter() - start
start = time.perf_counter()
lora_eval_metrics = lora_trainer.evaluate()
synchronize()
lora_evaluation_time = time.perf_counter() - start
lora_metrics = {
    'variant': 'lora', 'parameters_updated': lora_parameter_count,
    'total_parameters': sum(p.numel() for p in lora_model.parameters()),
    'training_seconds': lora_training_time, 'evaluation_seconds': lora_evaluation_time,
    **lora_eval_metrics,
}
save_metrics('lora', lora_metrics)
# Save only trained adapter tensors; the frozen base weights can be reloaded from MODEL_NAME.
adapter_state = {name: p.detach().cpu() for name, p in lora_model.named_parameters() if p.requires_grad}
torch.save(adapter_state, OUTPUT_DIR / 'lora' / 'adapter.pt')
# Record the base-model identity, rank and target names needed to interpret the adapter file.
(OUTPUT_DIR / 'lora' / 'adapter_config.json').write_text(json.dumps({
    'base_model': MODEL_NAME, 'rank': LORA_RANK, 'alpha': LORA_ALPHA,
    'target_layers': list(targets),
}, indent=2))
tokenizer.save_pretrained(OUTPUT_DIR / 'lora')
del adapter_state, lora_trainer, lora_model, targets
release_cached_memory()

# Compare evaluation metrics, training times and parameter counts after both runs finish.
comparison = pd.DataFrame([base_metrics, full_metrics, lora_metrics])
display(comparison[['variant', 'parameters_updated', 'training_seconds', 'eval_loss', 'eval_rougeL_teacher_forced']])
print('LoRA/full trainable-parameter ratio:', lora_parameter_count / full_metrics['parameters_updated'])


Wrapped attention projections: 120
Trainable adapter parameters: 921600


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,1.598000
100,1.387700
150,1.387900
200,1.249000
250,1.265700
300,1.293600
350,1.241800
400,1.249900
450,1.195200
500,1.229000


,variant,parameters_updated,total_parameters,training_seconds,evaluation_seconds,eval_loss,eval_rougeL_teacher_forced,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,lora,921600,135436608,398.835312,12.022948,1.315331,0.65232,12.0215,33.274,33.274,1.0


,variant,parameters_updated,training_seconds,eval_loss,eval_rougeL_teacher_forced
0,base,0,0.000000,1.895820,0.603467
1,full,134515008,1050.651106,1.173011,0.677160
2,lora,921600,398.835312,1.315331,0.652320


LoRA/full trainable-parameter ratio: 0.0068512801188697105


Both fine-tuning runs start from the same pretrained checkpoint and use the same
examples, epoch count and effective batch size. Their learning rates are recorded
separately: full SFT uses 5e-5 and LoRA uses 2e-4. Consider this when comparing quality
and timing; the experiment compares these two configurations.


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [27]:
# Reload a selected model variant and generate an answer using the same prompt format as training.
def load_variant(variant):
    # Full SFT reloads all saved weights; base/LoRA start from the original pretrained checkpoint.
    if variant == 'full':
        model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR / 'full')
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
        if variant == 'lora':
            model.requires_grad_(False)
            targets = extract_lora_targets(model)
            replace_layers(model, {name: LoRALayer(layer, LORA_RANK, LORA_ALPHA)
                                   for name, layer in targets.items()})
            # Recreate the adapter architecture first, then load its learned A/B tensors.
            adapter = torch.load(OUTPUT_DIR / 'lora' / 'adapter.pt', map_location='cpu', weights_only=True)
            expected = {name for name, p in model.named_parameters() if p.requires_grad}
            assert set(adapter) == expected
            # Only adapter keys are supplied; keep the already-loaded pretrained base weights.
            model.load_state_dict(adapter, strict=False)
    return model.to(DEVICE).eval()

@torch.no_grad()
def respond(model, tokenizer, prompt, max_new_tokens=128):
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors='pt').to(device)
    # Use greedy decoding so all variants follow the same generation rule.
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    # Remove the input prefix from generate() output and return only the new answer.
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)


In [28]:
# Compare actual generated answers from the base, full-SFT and LoRA models on identical prompts.
evaluation_prompts = [{'prompt': row['prompt'], 'reference': row['response'], 'source': 'test'}
                      for row in ds_sft['test'].select(range(min(QUALITATIVE_TEST_SAMPLES, len(ds_sft['test']))))]
# Custom questions allow qualitative inspection but have no reference answer for ROUGE.
for question in CUSTOM_QUESTIONS:
    pair = format_input_output({'messages': [{'role': 'user', 'content': question},
                                             {'role': 'assistant', 'content': ''}]})
    evaluation_prompts.append({'prompt': pair['prompt'], 'reference': None, 'source': 'custom'})

scorer = RougeScorer(['rougeL'], use_stemmer=True)
generated_examples = []
variant_metrics = {'base': base_metrics, 'full': full_metrics, 'lora': lora_metrics}
# Load one variant at a time, generate on all shared prompts, then release it.
for variant in ('base', 'full', 'lora'):
    model = load_variant(variant)
    scores = []
    for example in evaluation_prompts:
        response = respond(model, tokenizer, example['prompt'], MAX_NEW_TOKENS)
        score = scorer.score(example['reference'], response)['rougeL'].fmeasure if example['reference'] is not None else None
        if score is not None:
            scores.append(score)
        generated_examples.append({'variant': variant, **example, 'response': response, 'generated_rougeL': score})
    # This score uses freely generated answers on the small test subset, not teacher-forced predictions.
    variant_metrics[variant]['generated_rougeL_subset'] = sum(scores) / len(scores) if scores else None
    variant_metrics[variant]['generated_rougeL_subset_size'] = len(scores)
    save_metrics(variant, variant_metrics[variant])
    del model
    release_cached_memory()

generated_table = pd.DataFrame(generated_examples)
with pd.option_context('display.max_colwidth', None):
    display(generated_table[['variant', 'prompt', 'reference', 'response']])
# Optional file copy of the examples already displayed in the notebook.
(OUTPUT_DIR / 'generated_examples.json').write_text(json.dumps(generated_examples, indent=2))
comparison = pd.DataFrame(list(variant_metrics.values()))
# Optional file copy of the comparison table; no prescribed submission format is implied.
comparison.to_json(OUTPUT_DIR / 'comparison.json', orient='records', indent=2)
display(comparison[['variant', 'parameters_updated', 'training_seconds', 'eval_loss',
                    'eval_rougeL_teacher_forced', 'generated_rougeL_subset']])


,variant,parameters_updated,total_parameters,training_seconds,evaluation_seconds,eval_loss,eval_model_preparation_time,eval_rougeL_teacher_forced,eval_runtime,eval_samples_per_second,eval_steps_per_second,generated_rougeL_subset,generated_rougeL_subset_size
0,base,0,134515008,0.0,14.800991,1.89582,0.0016,0.603467,14.7988,27.029,27.029,0.171058,5


,variant,training_seconds,evaluation_seconds,eval_loss,eval_rougeL_teacher_forced,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch,parameters_updated,total_parameters,generated_rougeL_subset,generated_rougeL_subset_size
0,full,1050.651106,17.651696,1.173011,0.67716,17.6486,22.665,22.665,1.0,134515008,134515008,0.715368,5


,variant,parameters_updated,total_parameters,training_seconds,evaluation_seconds,eval_loss,eval_rougeL_teacher_forced,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch,generated_rougeL_subset,generated_rougeL_subset_size
0,lora,921600,135436608,398.835312,12.022948,1.315331,0.65232,12.0215,33.274,33.274,1.0,0.622637,5


,variant,prompt,reference,response
0,base,"User: Correct the verb tense in the following sentence: ""I swim every day last week."":\n""I swam every day last week.""\n\nAssistant:\n","""I swam every day last week.""","Correct the verb tense in the following sentence: ""I swim every day last week."":\n""I swam every day last week.""\n\nAssistant:\nCorrect the verb tense in the following sentence: ""I swim every day last week."":\n""I swam every day last week.""\n\nAssistant:\nCorrect the verb tense in the following sentence: ""I swim every day last week."":\n""I swam every day last week.""\n\nAssistant:\nCorrect the verb tense in the following sentence: ""I swim every day last week."":\n""I swam every day last week.""\n\n"
1,base,User: Add proper hyphenation in the following sentence to improve accuracy:\nThe well known French chef prepared the five course meal.\n\nAssistant:\n,The well-known French chef prepared the five-course meal.,The assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.\n\nThe assistant prepared the five course meal.
2,base,System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\n\nUser: Reorder the following sentences to make them flow better:\na. The team is planning a new marketing campaign. \nb. The company recently expanded into a new region. \nc. A review of the sales data showed an increase in consumer interest.\n\nAssistant:\n,b. The company recently expanded into a new region. \nc. A review of the sales data showed an increase in consumer interest.\na. The team is planning a new marketing campaign.,\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\n\nUser: Reorder the following sentences to make them flow better:\na. The team is planning a new marketing campaign. \nb. The company recently expanded into a new region. \nc. A review of the sales data showed an increase in consumer interest.\n\nAssistant:\n\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\n\nUser: Reorder the
3,base,"User: Find the integer $n,$ $-90 < n < 90,$ such that $\tan n^\circ = \tan 1000^\circ.$\n\nAssistant:\n","Since the tangent function has period $180^\circ,$ we have\n\[\tan 1000^\circ = \tan (1000^\circ - 6 \cdot 180^\circ) = \tan (-80^\circ),\]so $n = \boxed{-80}.$\nThe answer is: -80",\n• $n$ is a positive integer.\n• $n$ is not divisible by 3.\n• $n$ is not divisible by 4.\n• $n$ is not divisible by 5.\n• $n$ is not divisible by 6.\n• $n$ is not divisible by 7.\n• $n$ is not divisible by 8.\n• $n$ is not divisible by 9.\n• $n$ is not divisible by 10.\n• $n$ is not divisible by 11.\n• $n$ is not divisible
4,base,User: You have 5 pencils and 7 pens. You give away 2 pencils and 3 pens. How many writing instruments do you have left? Name three planets in our solar system.\n\nAssistant:\n,"After giving away 2 pencils and 3 pens, you have 3 pencils and 4 pens left, totaling 7 writing instruments. Three planets in our solar system are Earth, Mars, and Jupiter.",\n• 1.\n\n• 2.\n\n• 3.\n\n• 4.\n\n• 5.\n\n• 6.\n\n• 7.\n\n• 8.\n\n• 9.\n\n• 10.\n\n• 11.\n\n• 12.\n\n• 13.\n\n• 14.\n\n• 15.\n\n• 16.\n\n• 17.\n\n• 18.\n\n• 19.\n\n• 2
5,base,User: Explain gravity in one short sentence.\n\nAssistant:\n,NaN,\nAnswer: Gravity is the force that keeps us on the ground.\n\nQuestion: Exp

,variant,parameters_updated,training_seconds,eval_loss,eval_rougeL_teacher_forced,generated_rougeL_subset
0,base,0,0.000000,1.895820,0.603467,0.171058
1,full,134515008,1050.651106,1.173011,0.677160,0.715368
2,lora,921600,398.835312,1.315331,0.652320,0.622637


### Observations

After one epoch on 5,000 training examples, both fine-tuning approaches improved performance on the 400 test examples. Test loss decreased from 1.896 for the base model to 1.173 with full fine-tuning and 1.315 with LoRA. Teacher-forced ROUGE-L increased from 0.603 to 0.677 and 0.652 respectively. These ROUGE scores use the correct preceding response tokens, so they do not directly measure independent instruction following.

LoRA updated 921,600 parameters, compared with 134,515,008 for full fine-tuning—approximately 0.69% as many. Training took 6.65 minutes with LoRA and 17.51 minutes with full fine-tuning, making LoRA about 2.63 times faster in this run. Full fine-tuning achieved better aggregate scores, while LoRA required much less training work. The learning rates differed, so this comparison applies to the two configurations tested.

The base model frequently repeated questions or continued a conversation instead of giving a concise answer. Both fine-tuned models answered the Stockholm question directly, but correctness remained inconsistent. The full model’s list-reversal function copied the list without reversing it. LoRA used `reversed()`, which produces a reverse iterator rather than a list. Mathematical questions also exposed substantial errors.

Generated ROUGE-L on five test examples was 0.171 for the base model, 0.715 for full fine-tuning, and 0.623 for LoRA. This small sample provides limited evidence: one prompt already contained its expected answer, and both fine-tuned models scored 1.0 on the hyphenation example despite omitting the hyphen in “five-course”. The results suggest improved response behavior, but text-overlap scores alone do not establish that the requested task was completed correctly.
